# Configuring and importing

In [19]:
from __future__ import annotations

import hashlib
import platform
import sys
import types
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve().parent

# Nếu đang chạy notebook ngay tại ROOT
if not (ROOT / "data").exists():
    ROOT = Path.cwd().resolve()

if not (ROOT / "data").exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục data tại {ROOT}"
    )

print(f"ROOT = {ROOT}")
print(f"DATA = {ROOT / 'data'}")

ROOT = D:\Academic\UET\ATCS\shock-detector
DATA = D:\Academic\UET\ATCS\shock-detector\data


In [ ]:
_C_RAW = ROOT / "data"
_C_INTERIM = ROOT / "data" / "interim"
_C_REPORTS = ROOT / "reports"
_C_FIGS = _C_REPORTS / "figures"
_C_TABLES = _C_REPORTS / "tables"
_C_LOGS = _C_REPORTS / "logs"
_C_G1_DIR = _C_LOGS
_C_LABELS_FILE = "MILKY_Recovery_Labels_202603_202608.csv"


_C_HORIZON = 5
_C_DEEP_M = 2 + _C_HORIZON
_C_END_CUT_HOURS = 24

_C_BRAKE_RATES = (0.10, 0.20, 0.30)
_C_RULE_K = (3, 5, 7, 10)
_C_F1_FEATURES = ["logk", "log_gap", "accel", "log_since_r1"]
_C_SEED = 42


_C_EXPECTED_TOTALS = {"rows": 14045, "baskets": 6081, "max_Ri": 55}
_C_EXPECTED_QUALITY = {"EXACT_UNIQUE": 9035, "EXACT_AMBIGUOUS_FIRST": 2573,
                       "REPORT_SECOND_ONLY": 2437}
_C_EXPECTED_CONT = {1: (0.41, 0.08, 6081), 2: (0.50, 0.15, 2523), 3: (0.70, 0.23, 1272),
                    5: (0.79, 0.31, 632), 7: (0.80, 0.32, 372), 10: (0.79, 0.38, 197),
                    15: (0.81, 0.43, 74), 20: (0.94, 0.59, 32)}
_C_EXPECTED_MONTH = {"03": (67.5, 0.37, 0.78, 2.7, 5.0), "04": (54.9, 0.43, 0.79, 3.5, 5.7),
                     "05": (48.8, 0.44, 0.75, 2.6, 6.0), "06": (46.6, 0.41, 0.84, 3.8, 6.1),
                     "07": (31.7, 0.44, 0.84, 3.4, 7.8), "08": (43.7, 0.44, 0.77, 3.9, 6.7)}

C = types.SimpleNamespace(
    ROOT=ROOT, RAW=_C_RAW, INTERIM=_C_INTERIM, REPORTS=_C_REPORTS, FIGS=_C_FIGS,
    TABLES=_C_TABLES, LOGS=_C_LOGS, G1_DIR=_C_G1_DIR, LABELS_FILE=_C_LABELS_FILE,
    HORIZON=_C_HORIZON, DEEP_M=_C_DEEP_M, END_CUT_HOURS=_C_END_CUT_HOURS,
    BRAKE_RATES=_C_BRAKE_RATES, RULE_K=_C_RULE_K, F1_FEATURES=_C_F1_FEATURES, SEED=_C_SEED,
    EXPECTED_TOTALS=_C_EXPECTED_TOTALS, EXPECTED_QUALITY=_C_EXPECTED_QUALITY,
    EXPECTED_CONT=_C_EXPECTED_CONT, EXPECTED_MONTH=_C_EXPECTED_MONTH,
)
print("C.RAW =", C.RAW)

C.RAW = D:\Academic\UET\ATCS\shock-detector\data


Data valid

In [ ]:
REQUIRED = ["timestamp", "basket_id", "side", "Ri"]
NS_PER_MIN = 60 * 10**9


def load_labels(path):
    df = pd.read_csv(path, sep=";", encoding="utf-8-sig")
    if df.shape[1] == 1:
        df = pd.read_csv(path, sep=",", encoding="utf-8-sig")
    miss = [c for c in REQUIRED if c not in df.columns]
    if miss:
        raise ValueError(f"Thiếu cột {miss}; cột hiện có: {list(df.columns)}")
    ts = pd.to_datetime(df["timestamp"], errors="coerce")
    if ts.isna().any():
        raise ValueError(f"{int(ts.isna().sum())} timestamp không parse được")
    df["timestamp"] = ts.astype("datetime64[ns]")
    df["Ri"] = pd.to_numeric(df["Ri"].astype(str).str.extract(r"(\d+)")[0]).astype(int)
    df["side"] = df["side"].astype(str).str.upper().str.strip()
    df["basket_id"] = df["basket_id"].astype(str).str.strip()
    if "timestamp_quality" not in df.columns:
        df["timestamp_quality"] = "UNKNOWN"
    return df.sort_values(["basket_id", "Ri"]).reset_index(drop=True)


def basket_table(df):
    g = df.groupby("basket_id")
    b = g.agg(side=("side", "first"), M=("Ri", "max"),
              t_start=("timestamp", "min"), t_end=("timestamp", "max"))
    b["month"] = b["t_start"].dt.to_period("M").astype(str)
    b["dur_h"] = (b["t_end"] - b["t_start"]).dt.total_seconds() / 3600
    return b


def same_side_overlaps(b):
    n = 0
    for _, d in b.groupby("side"):
        d = d.sort_values("t_start")
        run_max_end = np.maximum.accumulate(d["t_end"].values)
        n += int((d["t_start"].values[1:] < run_max_end[:-1]).sum())
    return n


def validate_labels(df, b):
    g = df.groupby("basket_id")["Ri"]
    n, mx, mn = g.size(), g.max(), g.min()
    gap = df.groupby("basket_id")["timestamp"].diff().dt.total_seconds()
    return [
        ("NaN trong cột bắt buộc", int(df[REQUIRED].isna().sum().sum()), ""),
        ("Trùng (basket_id, Ri)", int(df.duplicated(["basket_id", "Ri"]).sum()), ""),
        ("Basket thiếu bậc (Ri không phải 1..n)", int(((mx != n) | (mn != 1)).sum()), ""),
        ("Basket có nhiều hơn 1 chiều", int((df.groupby("basket_id")["side"].nunique() > 1).sum()), ""),
        ("Timestamp giảm theo Ri", int((gap < 0).sum()), "kỳ vọng 0"),
        ("Basket cùng chiều chồng thời gian", same_side_overlaps(b),
         "tính theo [R1, Ri cuối]; kỳ vọng 0"),
    ]


def continuation_table(b, ks=(1, 2, 3, 5, 7, 10, 15, 20)):
    rows = []
    for k in ks:
        reach = b["M"] >= k
        n = int(reach.sum())
        if n == 0:
            continue
        rows.append(dict(k=k, n=n,
                         p_ge1=float((b.loc[reach, "M"] >= k + 1).mean()),
                         p_ge5=float((b.loc[reach, "M"] >= k + 5).mean())))
    return pd.DataFrame(rows)


def build_samples(df, horizon=C.HORIZON, end_cut_hours=C.END_CUT_HOURS, eps=0.1):
    """Mỗi dòng = một lần bot sắp mở bậc Rk (k >= 2). Chỉ dùng thông tin có tại t_k.

    Họ 1 (trạng thái basket): logk, log_gap, accel, log_since_r1.
    Proxy từ nhãn (dự phòng nếu thiếu M1, KHÔNG dùng trong mô hình chính):
    n_rec_60m, opp_depth, opp_log_age. Các proxy này chỉ dùng sự kiện xảy ra TRƯỚC t_k.
    """
    d = df.sort_values(["basket_id", "Ri"]).copy()
    d["timestamp"] = d["timestamp"].astype("datetime64[ns]")
    g = d.groupby("basket_id")
    d["M"] = g["Ri"].transform("max")
    d["t1"] = g["timestamp"].transform("min")
    d["t_end"] = g["timestamp"].transform("max")
    d["gap_min"] = g["timestamp"].diff().dt.total_seconds() / 60
    d["gap_prev_min"] = d.groupby("basket_id")["gap_min"].shift(1)

    ts = d["timestamp"].values.astype("int64")
    all_ts = np.sort(ts)
    d["n_rec_60m"] = (np.searchsorted(all_ts, ts, "left")
                      - np.searchsorted(all_ts, ts - 60 * NS_PER_MIN, "left"))

    opp_depth = np.zeros(len(d))
    opp_age = np.full(len(d), 1440.0)
    sides = d["side"].values
    for s, o in (("BUY", "SELL"), ("SELL", "BUY")):
        ev = d[d["side"] == o].sort_values("timestamp")
        if ev.empty:
            continue
        ev_ts = ev["timestamp"].values.astype("int64")
        ev_ri = ev["Ri"].values
        m = sides == s
        idx = np.searchsorted(ev_ts, ts[m], "left") - 1
        ok = idx >= 0
        dep = np.where(ok, ev_ri[np.clip(idx, 0, None)], 0)
        age = np.where(ok, (ts[m] - ev_ts[np.clip(idx, 0, None)]) / NS_PER_MIN, 1440.0)
        opp_depth[m] = dep
        opp_age[m] = np.minimum(age, 1440.0)
    d["opp_depth"] = opp_depth
    d["opp_log_age"] = np.log1p(opp_age)

    data_end = d["timestamp"].max()
    cut = d["t_end"] > data_end - pd.Timedelta(hours=end_cut_hours)
    n_cut = int(d.loc[cut, "basket_id"].nunique())

    s = d[(d["Ri"] >= 2) & ~cut].copy()
    s["k"] = s["Ri"]
    s["y"] = ((s["M"] - s["k"]) >= horizon).astype(int)
    s["logk"] = np.log(s["k"])
    s["log_gap"] = np.log1p(s["gap_min"])
    s["accel"] = np.log((s["gap_min"] + eps) / (s["gap_prev_min"] + eps)).fillna(0.0)
    s["log_since_r1"] = np.log1p((s["timestamp"] - s["t1"]).dt.total_seconds() / 60)
    s["month"] = s["timestamp"].dt.to_period("M").astype(str)
    s["day"] = s["timestamp"].dt.floor("D")
    cols = ["basket_id", "side", "k", "timestamp", "month", "day", "M", "t_end", "y",
            "logk", "log_gap", "accel", "log_since_r1",
            "n_rec_60m", "opp_depth", "opp_log_age", "timestamp_quality"]
    info = dict(n_baskets_cut=n_cut, data_end=data_end,
                n_rows_all_k2=int((d["Ri"] >= 2).sum()), n_rows=len(s))
    return s[cols].reset_index(drop=True), info

He thong danh gia va so sanh cac mo hinh du doan

In [ ]:
def make_fold(samples, month):
    """Train = dòng của các tháng trước, loại basket kết thúc sau mốc cắt (purge).
    Test = dòng thuộc tháng `month`."""
    ms = pd.Timestamp(month + "-01")
    train = samples[samples["t_end"] < ms]
    test = samples[samples["month"] == month]
    n_purged = int(((samples["timestamp"] < ms) & (samples["t_end"] >= ms)).sum())
    return train, test, n_purged


def walk_forward(samples, test_months, features=None, rates=C.BRAKE_RATES, C_reg=1.0):
    """features=None -> luật theo độ sâu (score = k). Ngược lại logistic L2.
    Ngưỡng cho từng tỷ lệ phanh được chọn trên TRAIN, không dùng test."""
    outs = []
    for m in test_months:
        tr, te, _ = make_fold(samples, m)
        if len(te) == 0 or len(tr) == 0:
            continue
        if features is None:
            s_tr, s_te = tr["k"].to_numpy(float), te["k"].to_numpy(float)
        else:
            pipe = make_pipeline(StandardScaler(), LogisticRegression(C=C_reg, max_iter=1000))
            pipe.fit(tr[features], tr["y"])
            s_tr = pipe.predict_proba(tr[features])[:, 1]
            s_te = pipe.predict_proba(te[features])[:, 1]
        o = te[["basket_id", "k", "y", "M", "month", "day"]].copy()
        o["score"] = s_te
        for q in rates:
            o[f"thr_{round(q * 100)}"] = np.quantile(s_tr, 1 - q)
        o["fold"] = m
        outs.append(o)
    return pd.concat(outs, ignore_index=True)


def pr_auc(y, s):
    y = np.asarray(y)
    return float("nan") if y.sum() == 0 else float(average_precision_score(y, s))


def paired_bootstrap(y, s_a, s_b, groups, n_boot=1000, seed=C.SEED):
    """Bootstrap theo cụm (ngày), ghép cặp hai mô hình trên cùng mẫu lặp."""
    y, s_a, s_b = map(np.asarray, (y, s_a, s_b))
    codes, uniq = pd.factorize(np.asarray(groups))
    idx_by = [np.flatnonzero(codes == i) for i in range(len(uniq))]
    rng = np.random.default_rng(seed)
    a, b = [], []
    for _ in range(n_boot):
        pick = rng.integers(0, len(uniq), len(uniq))
        idx = np.concatenate([idx_by[i] for i in pick])
        if y[idx].sum() == 0:
            continue
        a.append(average_precision_score(y[idx], s_a[idx]))
        b.append(average_precision_score(y[idx], s_b[idx]))
    a, b = np.array(a), np.array(b)
    ci = lambda x: (float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5)))
    return dict(ap_a=pr_auc(y, s_a), ci_a=ci(a), ap_b=pr_auc(y, s_b), ci_b=ci(b),
                delta=pr_auc(y, s_a) - pr_auc(y, s_b), ci_delta=ci(a - b),
                frac_delta_pos=float((a - b > 0).mean()), n_boot=len(a))


def brake_sim(oos, thr):
    """HALT tại dòng đầu tiên có score >= thr trong mỗi basket (tuần tự).
    avoided = M - k + 1: số bậc Rk..RM không được mở. Đây là CẬN TRÊN, không phải giảm lỗ.
    'Bắt kịp' = HALT đầu tiên rơi vào dòng y = 1 (còn >= 5 bậc nữa)."""
    d = oos[["basket_id", "k", "y", "M", "score"]].copy()
    d["flag"] = d["score"].to_numpy() >= np.asarray(thr)
    b = d.groupby("basket_id", sort=False).agg(M=("M", "first"))
    first = (d[d["flag"]].sort_values(["basket_id", "k"]).groupby("basket_id", sort=False)
             .first()[["k", "y"]].rename(columns={"k": "k_halt", "y": "y_halt"}))
    b = b.join(first)
    halted = b["k_halt"].notna()
    deep = b["M"] >= C.DEEP_M
    avoided = (b["M"] - b["k_halt"] + 1).where(halted, 0)
    n_b, n_deep = len(b), int(deep.sum())
    caught = int(((b["y_halt"] == 1) & deep).sum())
    false_h = int((b["y_halt"] == 0).sum())
    return dict(
        brake_rate_rows=float(d["flag"].mean()),
        n_baskets=n_b, n_deep=n_deep,
        halted_pct=float(halted.mean()),
        deep_caught=caught,
        deep_caught_pct=caught / n_deep if n_deep else float("nan"),
        avoided_total=int(avoided.sum()),
        avoided_pct=float(avoided.sum() / max((b["M"] - 1).sum(), 1)),
        avoided_deep=int(avoided[deep].sum()),
        false_flag_rows=int((d["flag"] & (d["y"] == 0)).sum()),
        false_halts=false_h,
        false_halts_per100=100 * false_h / n_b,
    )

In [ ]:
class Report:
    def __init__(self):
        self.lines = []

    def p(self, s=""):
        print(s)
        self.lines.append(str(s))

    def h(self, title):
        self.p(f"\n{'=' * 70}\n## {title}\n{'=' * 70}")

    def df(self, d, **kw):
        self.p(d.to_string(**kw))

    def save(self, path):
        Path(path).write_text("\n".join(self.lines), encoding="utf-8")


def _mark(a, b, tol):
    return "OK" if abs(a - b) <= tol else "!!"


def _save(df, name, index=False):
    C.TABLES.mkdir(parents=True, exist_ok=True)
    df.to_csv(C.TABLES / f"g1_{name}.csv", index=index)

Kiem dinh du lieu va phan tich ket qua

In [ ]:
def sec_env(R, path, df):
    R.h("0. MÔI TRƯỜNG VÀ FILE")
    p = Path(path)
    R.p(f"python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} "
        f"| sklearn {sklearn.__version__}")
    R.p(f"file: {p.name} | {p.stat().st_size:,} bytes | md5 {hashlib.md5(p.read_bytes()).hexdigest()[:12]}")
    R.p(f"cột: {list(df.columns)}")
    R.p("2 dòng đầu sau khi parse:")
    R.df(df.head(2))


def sec_validation(R, df, b, check_expected):
    R.h("A. KIỂM TRA DỮ LIỆU VÀ ĐỐI CHIẾU project_context.md")
    R.p(f"Khoảng thời gian: {df['timestamp'].min()} -> {df['timestamp'].max()}")
    R.p(f"BUY/SELL (basket): {b['side'].value_counts().to_dict()}")
    R.p("\n[A1] Kiểm tra tính sạch")
    v = pd.DataFrame(validate_labels(df, b), columns=["check", "n_loi", "ghi_chu"])
    R.df(v, index=False)

    R.p("\n[A2] Tổng số")
    q = df["timestamp_quality"].value_counts()
    rows = [("rows", len(df), C.EXPECTED_TOTALS["rows"]),
            ("baskets", len(b), C.EXPECTED_TOTALS["baskets"]),
            ("max_Ri", int(b["M"].max()), C.EXPECTED_TOTALS["max_Ri"])]
    rows += [(k, int(q.get(k, 0)), v_) for k, v_ in C.EXPECTED_QUALITY.items()]
    t = pd.DataFrame(rows, columns=["metric", "computed", "expected"])
    t["check"] = np.where(t["computed"] == t["expected"], "OK", "!!") if check_expected else "-"
    R.df(t, index=False)
    ok_tot = bool((t["computed"] == t["expected"]).all())

    R.p("\n[A3] Theo tháng (basket tính theo tháng của R1; gap tính theo tháng của dòng)")
    d = df.sort_values(["basket_id", "Ri"]).copy()
    d["gap_min"] = d.groupby("basket_id")["timestamp"].diff().dt.total_seconds() / 60
    d["month"] = d["timestamp"].dt.to_period("M").astype(str)
    gm = d[d["Ri"] >= 2].groupby("month")["gap_min"].median()
    t0, t1 = df["timestamp"].min().normalize(), df["timestamp"].max().normalize()
    recs = []
    for m, g in b.groupby("month"):
        mm = m[-2:]
        mdays = pd.date_range(max(pd.Timestamp(m + "-01"), t0),
                              min(pd.Timestamp(m + "-01") + pd.offsets.MonthEnd(0), t1))
        act = g["t_start"].dt.normalize().nunique()
        r5 = g[g["M"] >= 5]
        recs.append(dict(month=m, baskets=len(g), per_day_active=len(g) / act,
                         per_day_calendar=len(g) / len(mdays),
                         P_R1_R2=(g["M"] >= 2).mean(),
                         P_R5_R6=(r5["M"] >= 6).mean() if len(r5) else np.nan,
                         reach_R10_pct=100 * (g["M"] >= 10).mean(),
                         med_gap_min=gm.get(m, np.nan),
                         BUY=int((g["side"] == "BUY").sum()), SELL=int((g["side"] == "SELL").sum()),
                         max_Ri=int(g["M"].max())))
    mt = pd.DataFrame(recs)
    R.df(mt.round(3), index=False)
    _save(mt, "monthly")
    if check_expected:
        R.p("Đối chiếu (|chênh| trong dung sai làm tròn của project_context.md):")
        bad = 0
        for _, r in mt.iterrows():
            e = C.EXPECTED_MONTH.get(r["month"][-2:])
            if e is None:
                continue
            day_ok = min(abs(r["per_day_active"] - e[0]), abs(r["per_day_calendar"] - e[0])) <= 0.6
            flags = [day_ok, abs(r["P_R1_R2"] - e[1]) <= .006, abs(r["P_R5_R6"] - e[2]) <= .006,
                     abs(r["reach_R10_pct"] - e[3]) <= .06, abs(r["med_gap_min"] - e[4]) <= .06]
            bad += flags.count(False)
            R.p(f"  {r['month']}: " + " ".join(f"{n}={'OK' if f else '!!'}" for n, f in
                zip(["per_day", "P12", "P56", "R10", "gap"], flags)))
        R.p(f"  -> số ô lệch: {bad}. Lệch 'per_day'/'gap' có thể do khác định nghĩa ngày/gap, "
            f"hãy xem lại cách tính trong bản tổng hợp.")
    return ok_tot


def sec_continuation(R, b, check_expected):
    R.h("B. P(đi tiếp | đang ở Rk) — đối chiếu bảng trong project_context.md")
    ct = continuation_table(b, ks=tuple(C.EXPECTED_CONT))
    if check_expected:
        ct["p_ge1_exp"] = [C.EXPECTED_CONT[k][0] for k in ct["k"]]
        ct["p_ge5_exp"] = [C.EXPECTED_CONT[k][1] for k in ct["k"]]
        ct["n_exp"] = [C.EXPECTED_CONT[k][2] for k in ct["k"]]
        ct["check"] = [("OK" if (abs(r.p_ge1 - r.p_ge1_exp) <= .006 and abs(r.p_ge5 - r.p_ge5_exp) <= .006
                                 and r.n == r.n_exp) else "!!") for r in ct.itertuples()]
    R.df(ct.round(3), index=False)
    _save(ct, "continuation")
    fig, ax = plt.subplots(figsize=(6, 4))
    kk = continuation_table(b, ks=tuple(range(1, 31)))
    ax.plot(kk["k"], kk["p_ge1"], "o-", label="P(thêm ≥ 1 bậc)")
    ax.plot(kk["k"], kk["p_ge5"], "s-", label="P(thêm ≥ 5 bậc)")
    ax.set_xlabel("Bậc hiện tại k"); ax.set_ylabel("Xác suất"); ax.legend(); ax.grid(alpha=.3)
    ax.set_title("Basket càng sâu càng dễ sâu thêm")
    fig.tight_layout(); C.FIGS.mkdir(parents=True, exist_ok=True)
    fig.savefig(C.FIGS / "g1_continuation.png", dpi=130); plt.close(fig)
    if not check_expected:
        return True
    return bool((ct["check"] == "OK").all())


def sec_depth(R, b):
    R.h("C. ĐỘ SÂU TỐI ĐA M — có dấu hiệu bot giới hạn depth không?")
    vc = b["M"].value_counts().sort_index()
    R.p(f"max M = {int(b['M'].max())}, số basket đạt max = {int(vc.iloc[-1])}")
    tail = vc[vc.index >= 10]
    R.p("Phân bố M >= 10 (M:số basket): " + ", ".join(f"{k}:{v}" for k, v in tail.items()))
    flagged = []
    for m, c in vc[vc.index >= 12].items():
        nb = [vc.get(m + j, 0) for j in (-2, -1, 1, 2) if (m + j) in vc.index]
        if nb and c >= 5 and c >= 2.5 * np.mean(nb):
            flagged.append((int(m), int(c), round(float(np.mean(nb)), 1)))
    R.p(f"Điểm dồn cục nghi ngờ (M, count, TB lân cận): {flagged if flagged else 'không thấy'}")
    R.p("Nếu có dồn cục: các basket tại M đó là nhãn bị cắt (censored) — ghi vào hạn chế.")
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(vc.index, vc.values); ax.set_yscale("log")
    ax.set_xlabel("Bậc cuối M"); ax.set_ylabel("Số basket (log)")
    ax.set_title("Phân bố độ sâu tối đa của basket"); ax.grid(alpha=.3, axis="y")
    fig.tight_layout(); fig.savefig(C.FIGS / "g1_depth_hist.png", dpi=130); plt.close(fig)


def sec_manual(R, df, b, seed):
    R.h("D. KIỂM TRA TAY 3 BASKET (M≈2, M≈8, M lớn nhất) — đối chiếu với bảng gốc nếu cần")
    rng = np.random.default_rng(seed)
    picks = []
    for target in (2, 8):
        c = b[b["M"] == target]
        if c.empty:
            c = b.iloc[[int((b["M"] - target).abs().values.argmin())]]
        picks.append(c.index[int(rng.integers(len(c)))])
    picks.append(b["M"].idxmax())
    for bid in picks:
        g = df[df["basket_id"] == bid].sort_values("Ri").copy()
        M = int(g["Ri"].max())
        g["gap_min"] = g["timestamp"].diff().dt.total_seconds() / 60
        g["y_neu_Rk"] = (M - g["Ri"] >= C.HORIZON).astype(int)
        g.loc[g["Ri"] < 2, "y_neu_Rk"] = np.nan
        R.p(f"\nbasket {bid} | {g['side'].iloc[0]} | M={M}")
        show = g if len(g) <= 14 else pd.concat([g.head(6), g.tail(4)])
        show = show.assign(gap_min=show["gap_min"].round(2))
        R.df(show[["Ri", "timestamp", "gap_min", "y_neu_Rk"]], index=False)
        if len(g) > 14:
            R.p(f"  ... (đã ẩn {len(g) - 10} dòng giữa)")


def sec_samples(R, samples, info):
    R.h("E. BẢNG MẪU (k >= 2)")
    R.p(f"Số dòng k>=2 trước khi cắt: {info['n_rows_all_k2']:,} | sau cắt 24h cuối: {info['n_rows']:,}")
    R.p(f"Basket bị loại do bậc cuối nằm trong {C.END_CUT_HOURS}h cuối dữ liệu "
        f"({info['data_end']}): {info['n_baskets_cut']}")
    R.p(f"Tỷ lệ dương tổng: {samples['y'].mean():.4f} ({int(samples['y'].sum())} dòng) "
        f"| số basket có dòng dương: {samples.loc[samples['y'] == 1, 'basket_id'].nunique()}")
    samples = samples.assign(k_bucket=pd.cut(samples["k"], [1, 4, 9, 10**6], labels=["2-4", "5-9", ">=10"]))
    by_k = samples.groupby("k_bucket", observed=True)["y"].agg(n="size", pos="sum", prevalence="mean")
    R.p("\nTheo nhóm k:"); R.df(by_k.round(4))
    by_m = samples.groupby("month")["y"].agg(n="size", pos="sum", prevalence="mean")
    by_m["baskets_pos"] = samples[samples["y"] == 1].groupby("month")["basket_id"].nunique()
    R.p("\nTheo tháng:"); R.df(by_m.round(4))
    _save(by_k.reset_index(), "prevalence_by_k"); _save(by_m.reset_index(), "prevalence_by_month")
    months = sorted(samples["month"].unique())
    R.p("\nPurge ranh giới tháng (dòng thuộc basket kéo sang tháng test, bị loại khỏi train):")
    for m in months[1:]:
        tr, te, npg = make_fold(samples, m)
        R.p(f"  test {m}: train={len(tr):,} (purge {npg}) | test={len(te):,} | "
            f"test dương={int(te['y'].sum())}")


def _row(name, s):
    return dict(model=name, brake_rate_rows=s["brake_rate_rows"], halted_baskets=s["halted_pct"],
                deep_caught_pct=s["deep_caught_pct"], deep_caught=s["deep_caught"],
                avoided_total=s["avoided_total"], avoided_pct=s["avoided_pct"],
                avoided_deep=s["avoided_deep"], false_halts_per100=s["false_halts_per100"],
                false_halts=s["false_halts"], false_flag_rows=s["false_flag_rows"])

So sanh cac chien luoc HALT, mo phong phanh va kiem tra PRAUC

In [25]:
def sec_baselines(R, samples, n_boot, seed):
    months = sorted(samples["month"].unique())[1:]
    R.h("F. BASELINE — WALK-FORWARD, PR-AUC GỘP, MÔ PHỎNG PHANH")
    R.p(f"Test months: {months}. Train luôn là các tháng trước, có purge.")
    rule = walk_forward(samples, months, features=None)
    logk = walk_forward(samples, months, features=["logk"])
    f1 = walk_forward(samples, months, features=C.F1_FEATURES)
    y = logk["y"].to_numpy()
    R.p(f"\nOOS: {len(y):,} dòng | dương {int(y.sum())} | prevalence {y.mean():.4f}")

    R.p("\n[F1] PR-AUC gộp out-of-sample (CI 95% bootstrap theo ngày, 'prevalence' = mô hình ngẫu nhiên)")
    boot = paired_bootstrap(y, f1["score"], logk["score"], logk["day"], n_boot, seed)
    ap_rule = pr_auc(y, rule["score"])
    tab = pd.DataFrame([
        dict(model="ngau_nhien (prevalence)", ap=y.mean(), ci_lo=np.nan, ci_hi=np.nan),
        dict(model="luat score=k", ap=ap_rule, ci_lo=np.nan, ci_hi=np.nan),
        dict(model="logistic log(k)  [baseline]", ap=boot["ap_b"], ci_lo=boot["ci_b"][0], ci_hi=boot["ci_b"][1]),
        dict(model="logistic ho 1 day du", ap=boot["ap_a"], ci_lo=boot["ci_a"][0], ci_hi=boot["ci_a"][1]),
    ])
    R.df(tab.round(4), index=False); _save(tab, "pr_auc_pooled")
    R.p(f"ΔPR-AUC (họ 1 đầy đủ − log k) = {boot['delta']:+.4f}, CI95% [{boot['ci_delta'][0]:+.4f}, "
        f"{boot['ci_delta'][1]:+.4f}], tỷ lệ lần lặp Δ>0 = {boot['frac_delta_pos']:.2f} (n_boot={boot['n_boot']})")
    R.p("Lưu ý: logistic chỉ có log(k) cho cùng thứ hạng với luật score=k, nên hai PR-AUC phải gần bằng nhau.")

    R.p("\n[F2] PR-AUC theo tháng (độ ổn định)")
    rows = []
    for m in months:
        mk = logk["month"] == m
        rows.append(dict(month=m, n=int(mk.sum()), pos=int(logk.loc[mk, "y"].sum()),
                         prevalence=logk.loc[mk, "y"].mean(),
                         ap_logk=pr_auc(logk.loc[mk, "y"], logk.loc[mk, "score"]),
                         ap_f1=pr_auc(f1.loc[mk, "y"], f1.loc[mk, "score"])))
    pm = pd.DataFrame(rows); R.df(pm.round(4), index=False); _save(pm, "pr_auc_by_month")

    R.p("\n[F3] Mô phỏng phanh tuần tự (gộp 04–08). avoided = CẬN TRÊN số bậc không mở, KHÔNG phải giảm lỗ.")
    R.p("     deep_caught = basket sâu (M>=7) bị HALT ở dòng y=1; false_halts/100 = basket bị HALT ở dòng y=0 trên 100 basket.")
    recs = []
    for K in C.RULE_K:
        recs.append(_row(f"luat HALT khi k>={K}", brake_sim(rule, K)))
    for q in C.BRAKE_RATES:
        recs.append(_row(f"logk  @{int(q*100)}% (thr tren train)", brake_sim(logk, logk[f"thr_{round(q*100)}"].values)))
    for q in C.BRAKE_RATES:
        recs.append(_row(f"ho1  @{int(q*100)}% (thr tren train)", brake_sim(f1, f1[f"thr_{round(q*100)}"].values)))
    bt = pd.DataFrame(recs)
    pct = bt.copy()
    for c in ["brake_rate_rows", "halted_baskets", "deep_caught_pct", "avoided_pct"]:
        pct[c] = (100 * pct[c]).round(1)
    pct["false_halts_per100"] = pct["false_halts_per100"].round(2)
    R.df(pct, index=False); _save(bt, "brake_baselines")
    R.p("Do độ sâu rời rạc, tỷ lệ phanh thực tế của baseline có thể lệch khỏi mục tiêu 10/20/30% (xem cột brake_rate_rows).")

    # đường cong bậc tránh được - phanh nhầm
    rates = [r / 100 for r in range(5, 65, 5)]
    lg = walk_forward(samples, months, features=["logk"], rates=rates)
    ff = walk_forward(samples, months, features=C.F1_FEATURES, rates=rates)
    cur = []
    for K in range(2, 13):
        s = brake_sim(rule, K); cur.append(dict(model="luat k>=K", param=K, **s))
    for r in rates:
        n = f"thr_{round(r*100)}"
        cur.append(dict(model="logistic log(k)", param=r, **brake_sim(lg, lg[n].values)))
        cur.append(dict(model="logistic ho 1", param=r, **brake_sim(ff, ff[n].values)))
    cur = pd.DataFrame(cur)
    _save(cur[["model", "param", "brake_rate_rows", "avoided_pct", "avoided_total", "false_halts_per100",
               "deep_caught_pct"]], "brake_curve")
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for name, g in cur.groupby("model"):
        g = g.sort_values("false_halts_per100")
        ax.plot(g["false_halts_per100"], 100 * g["avoided_pct"], "o-", ms=4, label=name)
    ax.set_xlabel("Phanh nhầm (basket / 100 basket)"); ax.set_ylabel("Bậc tránh được (% bậc R2..RM, cận trên)")
    ax.legend(); ax.grid(alpha=.3); ax.set_title("Đường cong bậc tránh được – phanh nhầm (OOS 04–08)")
    fig.tight_layout(); fig.savefig(C.FIGS / "g1_brake_curve.png", dpi=130); plt.close(fig)
    return boot, tab


def sec_sanity(R, samples, seed):
    R.h("G. KIỂM TRA TỰ ĐỘNG")
    rng = np.random.default_rng(seed)
    y = samples["y"].to_numpy()
    aps = [pr_auc(y, rng.random(len(y))) for _ in range(20)]
    ok = abs(np.mean(aps) - y.mean()) < 0.02
    R.p(f"PR-AUC của điểm ngẫu nhiên: {np.mean(aps):.4f} vs prevalence {y.mean():.4f} -> {'OK' if ok else '!!'}")
    return ok

In [26]:
LABELS_PATH = str(C.RAW / C.LABELS_FILE)
N_BOOT = 1000
SEED = C.SEED
# --skip-expected: bỏ đối chiếu với số liệu trong project_context.md (dùng cho dữ liệu giả lập)
CHECK_EXPECTED = True
OUT_DIR = None   # None -> C.G1_DIR (giống mặc định của run())

In [27]:
out_dir = Path(OUT_DIR or C.G1_DIR)
for d in (out_dir, C.TABLES, C.FIGS, C.INTERIM):
    d.mkdir(parents=True, exist_ok=True)
R = Report()
R.p("G1 REPORT — Cơ chế phanh Recovery Basket XAUUSD M1")
df = load_labels(LABELS_PATH)
b = basket_table(df)

G1 REPORT — Cơ chế phanh Recovery Basket XAUUSD M1


In [28]:
sec_env(R, LABELS_PATH, df)


## 0. MÔI TRƯỜNG VÀ FILE
python 3.12.10 | pandas 3.0.6 | numpy 2.5.4 | sklearn 1.9.1
file: MILKY_Recovery_Labels_202603_202608.csv | 818,441 bytes | md5 d8aed529dd82
cột: ['timestamp', 'basket_id', 'side', 'Ri', 'timestamp_quality', 'tick_match_count']
2 dòng đầu sau khi parse:
                timestamp  basket_id side  Ri timestamp_quality  tick_match_count
0 2026-03-02 01:00:33.703  BMM8H21S1  BUY   1      EXACT_UNIQUE                 1
1 2026-03-02 01:05:06.266  BMM8H21S1  BUY   2      EXACT_UNIQUE                 1


In [29]:
ok_tot = sec_validation(R, df, b, CHECK_EXPECTED)


## A. KIỂM TRA DỮ LIỆU VÀ ĐỐI CHIẾU project_context.md
Khoảng thời gian: 2026-03-02 01:00:33.703000 -> 2026-08-31 21:01:08
BUY/SELL (basket): {'SELL': 3102, 'BUY': 2979}

[A1] Kiểm tra tính sạch
                                check  n_loi                            ghi_chu
               NaN trong cột bắt buộc      0                                   
                Trùng (basket_id, Ri)      0                                   
Basket thiếu bậc (Ri không phải 1..n)      0                                   
          Basket có nhiều hơn 1 chiều      0                                   
               Timestamp giảm theo Ri      0                          kỳ vọng 0
    Basket cùng chiều chồng thời gian      0 tính theo [R1, Ri cuối]; kỳ vọng 0

[A2] Tổng số
               metric  computed  expected check
                 rows     14045     14045    OK
              baskets      6081      6081    OK
               max_Ri        55        55    OK
         EXACT_UNIQUE      9035      9

In [30]:
ok_cont = sec_continuation(R, b, CHECK_EXPECTED)


## B. P(đi tiếp | đang ở Rk) — đối chiếu bảng trong project_context.md
 k    n  p_ge1  p_ge5  p_ge1_exp  p_ge5_exp  n_exp check
 1 6081  0.415  0.082       0.41       0.08   6081    OK
 2 2523  0.504  0.147       0.50       0.15   2523    OK
 3 1272  0.704  0.234       0.70       0.23   1272    OK
 5  632  0.790  0.312       0.79       0.31    632    OK
 7  372  0.801  0.317       0.80       0.32    372    OK
10  197  0.792  0.376       0.79       0.38    197    OK
15   74  0.811  0.432       0.81       0.43     74    OK
20   32  0.938  0.594       0.94       0.59     32    OK


In [31]:
sec_depth(R, b)


## C. ĐỘ SÂU TỐI ĐA M — có dấu hiệu bot giới hạn depth không?
max M = 55, số basket đạt max = 1
Phân bố M >= 10 (M:số basket): 10:41, 11:38, 12:22, 13:11, 14:11, 15:14, 16:6, 17:13, 18:4, 19:5, 20:2, 21:3, 22:4, 23:2, 24:2, 25:1, 26:1, 27:3, 28:1, 29:1, 32:1, 33:1, 34:1, 35:2, 36:1, 38:2, 41:1, 42:1, 46:1, 55:1
Điểm dồn cục nghi ngờ (M, count, TB lân cận): không thấy
Nếu có dồn cục: các basket tại M đó là nhãn bị cắt (censored) — ghi vào hạn chế.


In [32]:
sec_manual(R, df, b, SEED)


## D. KIỂM TRA TAY 3 BASKET (M≈2, M≈8, M lớn nhất) — đối chiếu với bảng gốc nếu cần

basket BMMPB9M1H | SELL | M=2
 Ri               timestamp  gap_min  y_neu_Rk
  1 2026-03-13 19:57:00.068      NaN       NaN
  2 2026-03-13 20:00:29.125     3.48       0.0

basket BMRK3RYTL | SELL | M=8
 Ri               timestamp  gap_min  y_neu_Rk
  1 2026-07-14 03:43:27.906      NaN       NaN
  2 2026-07-14 03:52:38.547     9.18       1.0
  3 2026-07-14 03:59:01.304     6.38       1.0
  4 2026-07-14 04:02:31.796     3.51       0.0
  5 2026-07-14 04:09:15.114     6.72       0.0
  6 2026-07-14 04:11:45.019     2.50       0.0
  7 2026-07-14 04:15:03.636     3.31       0.0
  8 2026-07-14 05:00:12.328    45.14       0.0

basket BMSFJA9ZX | SELL | M=55
 Ri               timestamp  gap_min  y_neu_Rk
  1 2026-08-05 03:36:26.204      NaN       NaN
  2 2026-08-05 03:41:33.286     5.12       1.0
  3 2026-08-05 03:53:04.433    11.52       1.0
  4 2026-08-05 03:55:19.009     2.24       1.0
  5 2026-08-05 04:00:0

In [33]:
samples, info = build_samples(df)
sec_samples(R, samples, info)
try:
    samples.to_parquet(C.INTERIM / "samples.parquet", index=False)
    R.p(f"\nĐã lưu {C.INTERIM / 'samples.parquet'}")
except Exception as e:  # thiếu pyarrow
    samples.to_csv(C.INTERIM / "samples.csv", index=False)
    R.p(f"\n(parquet lỗi: {e}) -> đã lưu samples.csv")


## E. BẢNG MẪU (k >= 2)
Số dòng k>=2 trước khi cắt: 7,964 | sau cắt 24h cuối: 7,889
Basket bị loại do bậc cuối nằm trong 24h cuối dữ liệu (2026-08-31 21:01:08): 66
Tỷ lệ dương tổng: 0.2697 (2128 dòng) | số basket có dòng dương: 366

Theo nhóm k:
             n  pos  prevalence
k_bucket                       
2-4       4643  881      0.1897
5-9       1999  650      0.3252
>=10      1247  597      0.4787

Theo tháng:
            n  pos  prevalence  baskets_pos
month                                      
2026-03  1729  421      0.2435           86
2026-04  1442  417      0.2892           64
2026-05  1333  330      0.2476           62
2026-06  1191  371      0.3115           56
2026-07   990  241      0.2434           48
2026-08  1204  348      0.2890           51

Purge ranh giới tháng (dòng thuộc basket kéo sang tháng test, bị loại khỏi train):
  test 2026-04: train=1,729 (purge 0) | test=1,442 | test dương=417
  test 2026-05: train=3,171 (purge 0) | test=1,333 | test dương=330
  test 2

In [34]:
boot, _ = sec_baselines(R, samples, N_BOOT, SEED)


## F. BASELINE — WALK-FORWARD, PR-AUC GỘP, MÔ PHỎNG PHANH
Test months: ['2026-04', '2026-05', '2026-06', '2026-07', '2026-08']. Train luôn là các tháng trước, có purge.

OOS: 6,160 dòng | dương 1707 | prevalence 0.2771

[F1] PR-AUC gộp out-of-sample (CI 95% bootstrap theo ngày, 'prevalence' = mô hình ngẫu nhiên)
                      model     ap  ci_lo  ci_hi
    ngau_nhien (prevalence) 0.2771    NaN    NaN
               luat score=k 0.4210    NaN    NaN
logistic log(k)  [baseline] 0.4286 0.3614 0.4883
       logistic ho 1 day du 0.4251 0.3605 0.4840
ΔPR-AUC (họ 1 đầy đủ − log k) = -0.0035, CI95% [-0.0111, +0.0061], tỷ lệ lần lặp Δ>0 = 0.27 (n_boot=1000)
Lưu ý: logistic chỉ có log(k) cho cùng thứ hạng với luật score=k, nên hai PR-AUC phải gần bằng nhau.

[F2] PR-AUC theo tháng (độ ổn định)
  month    n  pos  prevalence  ap_logk  ap_f1
2026-04 1442  417      0.2892   0.4294 0.4329
2026-05 1333  330      0.2476   0.3848 0.3936
2026-06 1191  371      0.3115   0.4376 0.4470
2026-07  990

In [35]:
ok_rand = sec_sanity(R, samples, SEED)


## G. KIỂM TRA TỰ ĐỘNG
PR-AUC của điểm ngẫu nhiên: 0.2715 vs prevalence 0.2697 -> OK


In [36]:
R.h("H. TIÊU CHÍ HOÀN THÀNH G1")
tag = lambda ok: ("OK" if ok else "!!") if CHECK_EXPECTED else "-"
R.p(f"[{tag(ok_tot)}] Tổng số khớp project_context.md" + ("" if CHECK_EXPECTED else " (bỏ qua)"))
R.p(f"[{tag(ok_cont)}] Bảng P(đi tiếp | Rk) khớp" + ("" if CHECK_EXPECTED else " (bỏ qua)"))
R.p(f"[{'OK' if ok_rand else '!!'}] PR-AUC ngẫu nhiên ≈ prevalence")
R.p("[OK] Baseline có PR-AUC gộp kèm CI và đường cong bậc tránh được – phanh nhầm (reports/figures/g1_brake_curve.png)")
R.p("[ ] Còn lại (làm tay): so sánh với bản tổng hợp theo tháng; kiểm tra tay 3 basket ở mục D")
R.save(out_dir / "g1_report.txt")
R.p(f"\nĐã ghi {out_dir / 'g1_report.txt'}  <- gửi file này lại cho tôi")


## H. TIÊU CHÍ HOÀN THÀNH G1
[OK] Tổng số khớp project_context.md
[OK] Bảng P(đi tiếp | Rk) khớp
[OK] PR-AUC ngẫu nhiên ≈ prevalence
[OK] Baseline có PR-AUC gộp kèm CI và đường cong bậc tránh được – phanh nhầm (reports/figures/g1_brake_curve.png)
[ ] Còn lại (làm tay): so sánh với bản tổng hợp theo tháng; kiểm tra tay 3 basket ở mục D

Đã ghi D:\Academic\UET\ATCS\shock-detector\reports\logs\g1_report.txt  <- gửi file này lại cho tôi
